# 04 - Dataset Visualization

Draw lesion bounding boxes on the original faces (sanity-checking that the stored coordinates map correctly), with examples from all four severity classes, and inspect the train/test split + 5-fold cross-validation balance.

> Read-only: the original split/fold assignments are reported, never changed.

In [1]:
# Make the src/ package importable from the notebooks/ folder (CPU-only, no install).
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
pd.set_option('display.max_columns', 30)

from acnevision.data import (
    eda_data, eda_images, eda_severity, eda_lesions, eda_boxes, eda_viz,
)

FIG = Path.cwd().parent / 'outputs' / 'figures'


In [2]:
# Load once: per-image table (+ real image sizes) and the per-box table.
df = eda_data.load_dataframe()
df = eda_data.attach_image_sizes(df)
boxes = eda_data.explode_boxes(df)
print('images:', len(df), '| lesion boxes:', len(boxes))
df.head(3)


images: 1457 | lesion boxes: 18983


,image_file,file_name,level,severity,lesion_count_label,num_lesions_annotated,lesion_boxes,split,fold_0_split,fold_1_split,fold_2_split,fold_3_split,fold_4_split,num_boxes,image_path,width,height,mode,img_format,missing,corrupted,aspect_ratio
0,acne_00000.jpg,levle0_0.jpg,0,mild,3,3,"{'xmin': [223, 217, 186], 'ymin': [290, 250, 5...",train,train,train,train,test,train,3,/media/mahabur/af33425f-321f-4960-a4d5-f823d20...,525,700,RGB,JPEG,False,False,0.750000
1,acne_00001.jpg,levle0_1.jpg,0,mild,3,3,"{'xmin': [394, 654, 930], 'ymin': [933, 2160, ...",train,train,train,test,train,train,3,/media/mahabur/af33425f-321f-4960-a4d5-f823d20...,3112,3456,RGB,JPEG,False,False,0.900463
2,acne_00002.jpg,levle0_10.jpg,1,moderate,20,20,"{'xmin': [357, 394, 314, 316, 265, 170, 203, 9...",train,train,train,train,test,train,20,/media/mahabur/af33425f-321f-4960-a4d5-f823d20...,478,460,RGB,JPEG,False,False,1.039130


## 6. Bounding-box visualization
Save >=20 overlays (5 per severity class) under `outputs/figures/bounding_boxes/`.

In [3]:
paths = eda_viz.save_box_samples(df, FIG / 'bounding_boxes', per_class=5)
print('wrote', len(paths), 'overlay images across all 4 severity classes')

wrote 20 overlay images across all 4 severity classes


In [4]:
# Preview one overlay inline to confirm boxes land on real lesions.
from PIL import Image
import matplotlib.pyplot as plt
sample = sorted(p for p in paths if 'moderate' in p.name)[0]
plt.figure(figsize=(6, 7)); plt.imshow(Image.open(sample)); plt.axis('off')
plt.title(sample.name); plt.show()

/tmp/ipykernel_2484335/1204143641.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title(sample.name); plt.show()


## 7. Dataset splitting analysis
The default `split` column plus the five `fold_*_split` columns. We only read them - assignments are never modified.

In [5]:
eda_data.split_columns(df)

['fold_0_split',
 'fold_1_split',
 'fold_2_split',
 'fold_3_split',
 'fold_4_split',
 'split']

In [6]:
# Train/test counts for each split column
eda_data.all_split_tables(df)

,split_column,train,test
0,fold_0_split,1165,292
1,fold_1_split,1165,292
2,fold_2_split,1165,292
3,fold_3_split,1165,292
4,fold_4_split,1165,292
5,split,1165,292


In [7]:
# Per-column detail with percentages (default split shown; change the name)
eda_data.split_table(df, 'split')

,split,count,percentage
0,train,1165,79.96
1,test,292,20.04


In [8]:
# Does the severity mix stay balanced across the default train/test split?
pd.crosstab(df['severity'], df['split'], normalize='columns').round(3)

split,test,train
severity,,
mild,0.353,0.352
moderate,0.435,0.434
severe,0.123,0.125
very_severe,0.089,0.088


**Takeaways:** every split column is a clean ~80/20 train/test partition (1165 / 292). The 5 folds enable cross-validation. Severity mix is broadly preserved across the split.